In [ ]:
SELECT
  MIN(cycle_hire.rental_id) AS min_rental_id,
  MAX(cycle_hire.rental_id) AS max_rental_id,
  COUNT(DISTINCT cycle_hire.rental_id) AS distinct_rental_id_count,
  COUNT(cycle_hire.rental_id) AS total_rental_id_count
FROM
  `bigquery-public-data`.`london_bicycles`.`cycle_hire` AS cycle_hire;

Rental ID Range:
40346508	127946561	
Total Count
83434866

Bike ID Range:
1	90233	
Unique Values
31929	
Total Count
83434851

In [ ]:
SELECT
  CASE
    WHEN cycle_hire.duration <= 60 THEN '<1 min'
    WHEN cycle_hire.duration > 60 AND cycle_hire.duration <= 300 THEN '1-5 mins'
    WHEN cycle_hire.duration > 300 AND cycle_hire.duration <= 1800 THEN '5 to 30 mins'
    WHEN cycle_hire.duration > 1800 AND cycle_hire.duration <= 3600 THEN '30min to 1 hour'
    WHEN cycle_hire.duration > 3600 AND cycle_hire.duration <= 36000 THEN '1 to 10 hours'
    WHEN cycle_hire.duration > 36000 AND cycle_hire.duration <= 86400 THEN '10 hours to 1 day'
    ELSE '> 1 day'
END
  AS duration_range,
  COUNT(cycle_hire.rental_id) AS total_rentals
FROM
  `bigquery-public-data`.`london_bicycles`.`cycle_hire` AS cycle_hire
GROUP BY
  duration_range
ORDER BY
  duration_range;

Trip duration statistics

1	<1 min	375568  0.45%
2	1-5 mins	9252310 11.09%
3	5 to 30 mins	64154740 76.90%
4	30min to 1 hour	6378033 7.64%
5	1 to 10 hours	3161138 3.79%
6	10 hours to 1 day	65944 0.08%
7	> 1 day	47133 0.056%

Total Count
83434866

Hyde Park Corner, Hyde Park
671,688
2
Belgrove Street , King's Cross
593,065
3
Waterloo Station 3, Waterloo
527,112
4
Albert Gate, Hyde Park
461,108
5
Black Lion Gate, Kensington Gardens
459,716
6
Waterloo Station 1, Waterloo
419,334
7
Wellington Arch, Hyde Park
392,889
8
Hop Exchange, The Borough
385,926
9
Wormwood Street, Liverpool Street
354,663
10
Triangle Car Park, Hyde Park
353,033
11
Duke Street Hill, London Bridge
334,976
12
Brushfield Street, Liverpool Street
324,478
13
Bethnal Green Road, Shoreditch
319,191
14
Aquatic Centre, Queen Elizabeth Olympic Park
306,440
15
Craven Street, Strand
293,522
16
Storey's Gate, Westminster
291,048
17
Park Lane , Hyde Park
282,810
18
Serpentine Car Park, Hyde Park
280,172
19
Palace Gate, Kensington Gardens
276,525
20
Tooley Street, Bermondsey
269,700

In [ ]:
SELECT
    cycle_hire.start_station_name,
    EXTRACT(HOUR FROM cycle_hire.start_date) AS hour_of_day,
    COUNT(*) AS rental_count
FROM
    `bigquery-public-data.london_bicycles.cycle_hire` AS cycle_hire
WHERE
    cycle_hire.start_station_name IN (
        'Hyde Park Corner, Hyde Park',
        "Belgrove Street , King's Cross",
        'Waterloo Station 3, Waterloo',
        'Albert Gate, Hyde Park',
        'Black Lion Gate, Kensington Gardens'
    )
GROUP BY
    cycle_hire.start_station_name,
    hour_of_day
ORDER BY
    cycle_hire.start_station_name,
    hour_of_day;

In [ ]:
SELECT
    cycle_hire.start_station_name,
    COUNT(*) AS rental_count
FROM
    `bigquery-public-data.london_bicycles.cycle_hire` AS cycle_hire
GROUP BY
    cycle_hire.start_station_name
ORDER BY
    rental_count DESC
LIMIT 20;

In [ ]:
SELECT
    cycle_hire.start_station_name,
    FORMAT_DATE('%A', DATE(cycle_hire.start_date)) AS day_of_week,
    EXTRACT(DAYOFWEEK FROM cycle_hire.start_date) AS day_num,
    COUNT(*) AS rental_count
FROM
    `bigquery-public-data.london_bicycles.cycle_hire` AS cycle_hire
WHERE
    cycle_hire.start_station_name IN (
        'Hyde Park Corner, Hyde Park',
        "Belgrove Street , King's Cross",
        'Waterloo Station 3, Waterloo',
        'Albert Gate, Hyde Park',
        'Black Lion Gate, Kensington Gardens'
    )
GROUP BY
    cycle_hire.start_station_name,
    day_of_week,
    day_num
ORDER BY
    cycle_hire.start_station_name,
    day_num;

In [ ]:
SELECT
  daily_hires.start_station_name,
  AVG(daily_hires.count_of_rentals) AS average_daily_hires
FROM (
  SELECT
    cycle_hire.start_station_name,
    DATE(cycle_hire.start_date) AS rental_date,
    COUNT(cycle_hire.rental_id) AS count_of_rentals
  FROM
    `bigquery-public-data`.`london_bicycles`.`cycle_hire` AS cycle_hire
    where date_trunc(start_date>'2019-12-31')
  GROUP BY
    cycle_hire.start_station_name,
    rental_date ) AS daily_hires
GROUP BY
  daily_hires.start_station_name
ORDER BY
  daily_hires.start_station_name;

In [ ]:
WITH monthly_counts AS (
  SELECT
    DATE_TRUNC(cycle_hire.start_date, MONTH) AS rental_month,
    COUNT(*) AS total_ebike_rentals
  FROM
    `bigquery-public-data.london_bicycles.cycle_hire` AS cycle_hire
  WHERE
    cycle_hire.bike_model = 'PBSC_EBIKE'
  GROUP BY
    1
)
SELECT
  rental_month,
  total_ebike_rentals,
  LAG(total_ebike_rentals) OVER (ORDER BY rental_month) AS previous_month_rentals,
  ROUND((total_ebike_rentals - LAG(total_ebike_rentals) OVER (ORDER BY rental_month)) * 100.0 / NULLIF(LAG(total_ebike_rentals) OVER (ORDER BY rental_month), 0), 2) AS percentage_change_mom
FROM
  monthly_counts
ORDER BY
  rental_month;

In [ ]:
-- SHOW THE top 20 most popular start stations in 2022 and 2023, with their total rental counts and average duration
SELECT
  cycle_hire.start_station_name,
  COUNT(cycle_hire.rental_id) AS total_rental_count,
  AVG(cycle_hire.duration) AS average_duration_seconds
FROM
  `bigquery-public-data`.`london_bicycles`.`cycle_hire` AS cycle_hire
WHERE
  TIMESTAMP_TRUNC(cycle_hire.start_date, YEAR) IN (TIMESTAMP '2022-01-01 00:00:00',
    TIMESTAMP '2023-03-01 00:00:00')
GROUP BY
  cycle_hire.start_station_name, bike_model
ORDER BY
  COUNT(cycle_hire.rental_id) DESC
LIMIT
  100;

In [ ]:
# ...existing code...
import requests
import pandas as pd
import io
import re

xml_url = "https://tfl.gov.uk/tfl/syndication/feeds/cycle-hire/livecyclehireupdates.xml"
resp = requests.get(xml_url, timeout=15)
resp.raise_for_status()

# attempt direct read; fallback to manual parsing if needed
try:
    df = pd.read_xml(io.BytesIO(resp.content), parser="lxml")
except Exception:
    import xml.etree.ElementTree as ET
    root = ET.fromstring(resp.content)
    # find the most common child tag (repeating records)
    tag_counts = {}
    for child in root:
        tag_counts[child.tag] = tag_counts.get(child.tag, 0) + 1
    rep_tag = max(tag_counts, key=tag_counts.get)
    rows = []
    for elem in root.findall(rep_tag):
        rows.append({c.tag: c.text for c in elem})
    df = pd.DataFrame(rows)

# removal of "'" in station names. Followed by split text by ",". 
cleanApst = df['name'].astype(str).str.replace("'", "", regex=False).str.strip()
df['name'] = cleanApst
df.insert(2, 'district', cleanApst.str.split(',', n=1).str[1].str.strip())

# fill na installation dates with 2015-01-01 00:00:00
df['installDate'] = df['installDate'].fillna('1420070400000')

df['installDate'] = pd.to_datetime(df['installDate'].astype(int), unit='ms').dt.strftime('%Y-%m-%d')
df['removalDate'] = pd.to_datetime(pd.to_numeric(df['removalDate'], errors='coerce'), unit='ms').dt.strftime('%Y-%m-%d')
print(df.shape)
df.head(10)
# output to a csv file on local drive
# Example Windows absolute path (using a raw string)
# windows_path = r'C:\Users\YourUser\Documents\sales_data.csv'
# df.to_csv(windows_path, index=False)
# Example macOS/Linux absolute path
# mac_linux_path = '/Users/YourUser/Documents/sales_data.csv'
# df.to_csv(mac_linux_path, index=False)
mac_path='/Users/lingluli/Desktop/NTU DSAI Course/DSAI-M2-Assignment/london_cycle_station_now.csv'
df.to_csv(mac_path)
# ...existing code...

(800, 16)
